# Post-hoc Proofreader Analysis from Tasks + DifferStack History

This notebook shows how to:

1. Query task-level proofreading outcomes from NeuVue Queue.
2. Retrieve Neuroglancer `differ_stack` entries (the same change history used by "Rewind history").
3. Normalize both datasets into analysis-ready tables.
4. Produce per-proofreader post-hoc summaries.

It is based on query patterns from:
- `neuvue-manage/task-generation/namespaces/generic-forced-choice-task/query_tasks.ipynb`
- `neuvueclient.NeuvueQueue` API methods in `neuvue_project/deps/neuvue-client/neuvueclient/__init__.py`


## Notes Before Running

- The notebook assumes you have credentials configured for `NeuvueQueue` auth.
- By default, `get_tasks(...)` converts Neuroglancer state URLs to JSON, which is slow. We disable this with `convert_states_to_json=False` for faster analytics queries.
- `differ_stack` is posted by workspace submissions that include `viewer.differ.stack`.
  - Some workflows (for example spelunker mode) may not send differstack data.
  - It is normal for some tasks to have no differstack entries.


In [ ]:
from __future__ import annotations

import os
import sys
import json
import datetime as dt
from pathlib import Path

import pandas as pd
import numpy as np


In [ ]:
# Make local neuvue-client importable from this repository checkout.
REPO_ROOT = Path.cwd().resolve()
CLIENT_PATH = REPO_ROOT / "neuvue_project" / "deps" / "neuvue-client"
if str(CLIENT_PATH) not in sys.path:
    sys.path.insert(0, str(CLIENT_PATH))

from neuvueclient import NeuvueQueue


## Configure Query Scope

Set these values for your analysis window.

Tips:
- Keep the first query narrow (single namespace + recent date range) to iterate quickly.
- Expand once the pipeline looks right.


In [ ]:
QUEUE_URL = os.getenv("NEUVUE_QUEUE_URL", "https://queue.neuvue.io")

NAMESPACE = "forcedChoiceExample"       # change me
ASSIGNEES = ["dxenes1"]                 # set [] for all assignees in namespace
TASK_STATUSES = ["closed", "errored"]  # typical submitted outcomes in this app

# Optional closed-time filter.
# Use None for open-ended range.
CLOSED_AFTER = dt.datetime(2024, 1, 1)
CLOSED_BEFORE = None

# Keep payload light for analytics; add more columns as needed.
TASK_SELECT = [
    "assignee",
    "author",
    "namespace",
    "status",
    "priority",
    "duration",
    "created",
    "opened",
    "closed",
    "metadata",
    "seg_id",
    "tags",
    "ng_state",
]


In [ ]:
client = NeuvueQueue(QUEUE_URL)
print(f"Connected to: {QUEUE_URL}")


## Build Task Sieve

This follows the same `sieve` style used in the example query notebook.


In [ ]:
def build_task_sieve(
    namespace: str,
    assignees: list[str] | None = None,
    statuses: list[str] | None = None,
    closed_after: dt.datetime | None = None,
    closed_before: dt.datetime | None = None,
) -> dict:
    sieve = {"namespace": namespace}

    if assignees:
        sieve["assignee"] = assignees[0] if len(assignees) == 1 else assignees

    if statuses:
        sieve["status"] = statuses[0] if len(statuses) == 1 else statuses

    if closed_after is not None or closed_before is not None:
        closed_query = {}
        if closed_after is not None:
            closed_query["$gt"] = closed_after
        if closed_before is not None:
            closed_query["$lt"] = closed_before
        sieve["closed"] = closed_query

    return sieve


task_sieve = build_task_sieve(
    namespace=NAMESPACE,
    assignees=ASSIGNEES,
    statuses=TASK_STATUSES,
    closed_after=CLOSED_AFTER,
    closed_before=CLOSED_BEFORE,
)

task_sieve


## Query Tasks

`convert_states_to_json=False` avoids expensive state download and is usually better for post-hoc behavior analysis.


In [ ]:
tasks = client.get_tasks(
    sieve=task_sieve,
    select=TASK_SELECT,
    convert_states_to_json=False,
    pageSize=1000,
)

print(f"Fetched tasks: {len(tasks)}")
tasks.head(3)


In [ ]:
if tasks.empty:
    raise RuntimeError("No tasks matched your sieve. Adjust namespace/assignee/date filters.")

# Move index to explicit column for joins.
tasks = tasks.reset_index(names="task_id")

# Common metadata fields used in forced-choice style workflows.
if "metadata" in tasks.columns:
    tasks["decision"] = tasks["metadata"].apply(
        lambda m: m.get("decision") if isinstance(m, dict) else None
    )
    tasks["flag_reason"] = tasks["metadata"].apply(
        lambda m: m.get("flag_reason") if isinstance(m, dict) else None
    )

tasks[[c for c in ["task_id", "assignee", "status", "duration", "decision", "closed"] if c in tasks.columns]].head(10)


## Quick Task-Level Sanity Checks

In [ ]:
display(tasks["status"].value_counts(dropna=False).rename("n_tasks"))

if "decision" in tasks.columns:
    display(tasks["decision"].value_counts(dropna=False).rename("n_tasks"))

display(
    tasks.groupby("assignee", dropna=False)
    .agg(
        n_tasks=("task_id", "nunique"),
        median_duration_s=("duration", "median"),
        mean_duration_s=("duration", "mean"),
    )
    .sort_values("n_tasks", ascending=False)
)


## Pull DifferStack History for These Tasks

The client supports listing differ stacks via `get_differ_stacks(...)`.
We query by task IDs in chunks so large analyses remain stable.


In [ ]:
def chunks(items: list[str], chunk_size: int):
    for i in range(0, len(items), chunk_size):
        yield items[i : i + chunk_size]


def get_differstacks_for_tasks(client: NeuvueQueue, task_ids: list[str], chunk_size: int = 200) -> pd.DataFrame:
    frames = []
    task_ids = [str(t) for t in task_ids]

    for task_chunk in chunks(task_ids, chunk_size):
        sieve = {"task_id": {"$in": task_chunk}, "active": True}
        try:
            ds = client.get_differ_stacks(sieve=sieve, pageSize=1000)
            if len(ds) > 0:
                frames.append(ds)
        except Exception as chunk_exc:
            # Fallback: query per task if your queue deployment does not support $in here.
            print(f"Chunk query failed ({chunk_exc}). Falling back to per-task queries for this chunk.")
            for tid in task_chunk:
                try:
                    ds_single = client.get_differ_stacks(sieve={"task_id": tid, "active": True}, pageSize=1000)
                    if len(ds_single) > 0:
                        frames.append(ds_single)
                except Exception as item_exc:
                    print(f"  task_id={tid}: {item_exc}")

    if not frames:
        return pd.DataFrame(columns=["_id", "task_id", "differ_stack"])  # consistent shape

    out = pd.concat(frames, axis=0)
    out = out[~out.index.duplicated(keep="first")]
    return out


differstacks = get_differstacks_for_tasks(client, tasks["task_id"].tolist(), chunk_size=200)
print(f"Fetched differstack rows: {len(differstacks)}")
differstacks.head(3)


## Normalize DifferStack Entries to an Event Table

Each `differ_stack` row is a list of change events captured during a proofreading session.
Because event payload shape can vary, we flatten defensively and infer action/time columns.


In [ ]:
def _coerce_event_to_dict(event):
    if isinstance(event, dict):
        return event
    if isinstance(event, list):
        return {"raw_event": event}
    if event is None:
        return {}
    return {"raw_event": event}


def _pick_first_existing(cols, candidates):
    for c in candidates:
        if c in cols:
            return c
    return None


def _parse_event_time(series: pd.Series) -> pd.Series:
    # Handles numeric epoch seconds/ms and string timestamps.
    numeric = pd.to_numeric(series, errors="coerce")

    if numeric.notna().mean() > 0.6:
        med = numeric.dropna().median() if numeric.notna().any() else np.nan
        unit = "ms" if pd.notna(med) and med > 1e11 else "s"
        return pd.to_datetime(numeric, unit=unit, errors="coerce", utc=True)

    return pd.to_datetime(series, errors="coerce", utc=True)


def normalize_differstack_events(differ_df: pd.DataFrame) -> pd.DataFrame:
    if differ_df.empty:
        return pd.DataFrame(columns=["differ_id", "task_id", "event_order", "action", "event_time"])

    temp = differ_df.reset_index(names="differ_id").copy()

    # Ensure list-like values; empty/None -> empty list so explode behaves predictably.
    temp["differ_stack"] = temp["differ_stack"].apply(
        lambda x: x if isinstance(x, list) else ([] if pd.isna(x) else [x])
    )

    exploded = temp.explode("differ_stack", ignore_index=True)
    exploded = exploded[exploded["differ_stack"].notna()].copy()
    exploded["event_order"] = exploded.groupby("differ_id").cumcount()

    payload = exploded["differ_stack"].apply(_coerce_event_to_dict)
    flat = pd.json_normalize(payload)

    events = pd.concat([
        exploded[["differ_id", "task_id", "event_order"]].reset_index(drop=True),
        flat.reset_index(drop=True),
    ], axis=1)

    action_candidates = [
        "action", "type", "event", "operation", "op", "kind", "name", "changeType", "change_type"
    ]
    time_candidates = [
        "timestamp", "time", "ts", "created", "created_at", "updated", "updated_at", "date"
    ]

    action_col = _pick_first_existing(events.columns, action_candidates)
    time_col = _pick_first_existing(events.columns, time_candidates)

    if action_col is not None:
        events["action"] = events[action_col].astype(str)
    else:
        events["action"] = "unknown"

    if time_col is not None:
        events["event_time"] = _parse_event_time(events[time_col])
    else:
        events["event_time"] = pd.NaT

    events["task_id"] = events["task_id"].astype(str)
    return events


events = normalize_differstack_events(differstacks)
print(f"Normalized events: {len(events)}")
print(f"Columns: {len(events.columns)}")
display(events.head(5))


In [ ]:
# Inspect available event keys so you can tune action/time parsing if needed.
# This is useful because differ payload schemas can vary by Neuroglancer build/workflow.

event_columns = [c for c in events.columns if c not in {"differ_id", "task_id", "event_order", "action", "event_time"}]
pd.Series(event_columns, name="event_columns")


## Join Events Back to Task Metadata

In [ ]:
task_cols_for_join = [
    c for c in [
        "task_id",
        "assignee",
        "author",
        "namespace",
        "status",
        "priority",
        "duration",
        "created",
        "opened",
        "closed",
        "decision",
        "flag_reason",
        "seg_id",
        "tags",
    ]
    if c in tasks.columns
]

events_enriched = events.merge(
    tasks[task_cols_for_join],
    on="task_id",
    how="left",
    validate="m:1",
)

display(events_enriched.head(5))


## Post-hoc Proofreader Summaries

The tables below are a practical starting point for behavior analysis:
- workload and timing per proofreader
- number of differ events per task and per proofreader
- action mix (if action field exists in your differ payload)


In [ ]:
events_per_task = (
    events_enriched.groupby("task_id", dropna=False)
    .size()
    .rename("n_events")
    .reset_index()
)

tasks_with_event_counts = tasks.merge(events_per_task, on="task_id", how="left")
tasks_with_event_counts["n_events"] = tasks_with_event_counts["n_events"].fillna(0).astype(int)

proofreader_summary = (
    tasks_with_event_counts.groupby("assignee", dropna=False)
    .agg(
        n_tasks=("task_id", "nunique"),
        n_tasks_with_differ=("n_events", lambda x: int((x > 0).sum())),
        median_duration_s=("duration", "median"),
        mean_duration_s=("duration", "mean"),
        mean_events_per_task=("n_events", "mean"),
        median_events_per_task=("n_events", "median"),
    )
    .sort_values("n_tasks", ascending=False)
)

display(proofreader_summary)


In [ ]:
if not events_enriched.empty:
    action_by_proofreader = (
        events_enriched.groupby(["assignee", "action"], dropna=False)
        .size()
        .rename("n_events")
        .reset_index()
        .sort_values(["assignee", "n_events"], ascending=[True, False])
    )
    display(action_by_proofreader.head(30))
else:
    print("No differ events found for the selected tasks.")


## Optional: Timeline View for a Single Task

Useful for deep-dives into one proofreader session.


In [ ]:
EXAMPLE_TASK_ID = tasks["task_id"].iloc[0]

task_timeline = (
    events_enriched.loc[events_enriched["task_id"] == EXAMPLE_TASK_ID]
    .sort_values(["event_time", "event_order"], na_position="last")
)

print(f"Example task_id: {EXAMPLE_TASK_ID}")
display(task_timeline.head(50))


## Export Analysis Tables

Write outputs so you can use them in downstream notebooks/scripts.


In [ ]:
OUT_DIR = REPO_ROOT / "notebooks" / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

tasks_with_event_counts.to_csv(OUT_DIR / "tasks_with_event_counts.csv", index=False)
events_enriched.to_csv(OUT_DIR / "differ_events_enriched.csv", index=False)
proofreader_summary.to_csv(OUT_DIR / "proofreader_summary.csv")

print(f"Wrote outputs to: {OUT_DIR}")


## Interpretation Ideas for Proofreader Action Analysis

- Compare `mean_events_per_task` vs `median_duration_s` to detect workflow differences across proofreaders.
- Stratify by `decision` to test whether certain outcomes require more interaction steps.
- Look at `flag_reason` and event count/action mix to characterize failure modes.
- Build task-level features (`n_events`, dominant action, first/last event times) for QC models.
